<a href="https://colab.research.google.com/github/CSUC/RDR-scripts/blob/main/delete_files/delete_files.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title First click the ▶ button to execute the script. Then, enter your API token and dataset DOI. Finally, choose whether to delete all files or a specific folder/subfolder.

import subprocess
import sys
import os

# ============================================================
# Install required packages
# ============================================================

def install_packages():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "-q"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyDataverse", "-q"])
    print("Libraries have been downloaded or updated.")

try:
    import pyDataverse
except ImportError:
    print("Installing libraries...")
    install_packages()

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import requests
import pandas as pd
import ipywidgets as widgets

from pyDataverse.api import NativeApi
from IPython.display import display, clear_output, HTML


# ============================================================
# Configuration
# ============================================================

BASE_URL = "https://dataverse.csuc.cat"

API_TOKEN = input("Please enter your API token: ").strip()
DOI = input("Please enter the dataset DOI in this format \"doi:10.34810/dataXXX\": ").strip()

headers = {
    "X-Dataverse-key": API_TOKEN
}

api = NativeApi(BASE_URL, API_TOKEN)


# ============================================================
# Functions
# ============================================================

def get_all_file_metadata(api, doi):
    response = api.get_dataset(doi)
    response.raise_for_status()

    files = response.json()["data"]["latestVersion"]["files"]

    rows = []

    for f in files:
        data_file = f.get("dataFile", {})

        rows.append({
            "id": data_file.get("id"),
            "filename": data_file.get("filename"),
            "directoryLabel": f.get("directoryLabel", "")
        })

    return pd.DataFrame(rows)


def get_files_for_folder(df_files, selected_folder):
    if selected_folder == "ALL DATASET FILES":
        return df_files.copy()

    selected_files = df_files[
        df_files["directoryLabel"].fillna("").apply(
            lambda x: x == selected_folder or x.startswith(selected_folder + "/")
        )
    ].copy()

    return selected_files


def delete_files(df_to_delete):
    deleted_count = 0
    failed_ids = []

    for file_id in df_to_delete["id"].dropna().astype(int):
        url = f"{BASE_URL}/api/files/{file_id}"

        print(f"Deleting file with id={file_id} ...")

        try:
            response = requests.delete(url, headers=headers)

            if response.status_code == 200:
                print(f"Deleted file {file_id}")
                deleted_count += 1

            else:
                print(
                    f"Failed to delete file {file_id} "
                    f"(HTTP {response.status_code})"
                )
                print(response.text)

                failed_ids.append(
                    (
                        file_id,
                        response.status_code,
                        response.text
                    )
                )

        except Exception as e:
            print(f"Error deleting file {file_id}: {e}")

            failed_ids.append(
                (
                    file_id,
                    "Exception",
                    str(e)
                )
            )

    print("\n========================================")
    print("SUMMARY")
    print("========================================")

    print(f"\nDeleted {deleted_count}/{len(df_to_delete)} files.")

    if failed_ids:
        print("\nFailed deletions:\n")

        for fid, status, msg in failed_ids:
            print(f" - ID {fid} | status: {status}")

    else:
        print("\nAll selected files were deleted successfully.")


# ============================================================
# Load dataset metadata
# ============================================================

print("\nLoading dataset metadata...")

df_files = get_all_file_metadata(api, DOI)

if df_files.empty:
    print("\nNo files found in dataset.")
else:
    print(f"\nFound {len(df_files)} files in dataset {DOI}")

    display(df_files[["id", "filename", "directoryLabel"]])

    folders = sorted(
        folder
        for folder in df_files["directoryLabel"].fillna("").unique()
        if folder != ""
    )

    options = ["ALL DATASET FILES"] + folders

    folder_widget = widgets.Dropdown(
        options=options,
        value="ALL DATASET FILES",
        description="Delete:",
        disabled=False,
        layout=widgets.Layout(width="80%")
    )

    preview_button = widgets.Button(
        description="Preview selected files",
        button_style="info"
    )

    delete_button = widgets.Button(
        description="Delete selected files",
        button_style="danger",
        disabled=True
    )

    confirmation_widget = widgets.Text(
        value="",
        placeholder="Type Yes to confirm deletion",
        description="Confirm:",
        disabled=False,
        layout=widgets.Layout(width="50%")
    )

    output_area = widgets.Output()

    selected_df = None

    def preview_selection(button):
        global selected_df

        with output_area:
            clear_output()

            selected_target = folder_widget.value
            selected_df = get_files_for_folder(df_files, selected_target)

            print("========================================")
            print("SELECTED TARGET")
            print("========================================")
            print(selected_target)

            print(f"\nFiles selected: {len(selected_df)}\n")

            display(selected_df[["id", "filename", "directoryLabel"]])

            print("\nTo delete these files, type exactly 'Yes' in the confirmation box.")
            delete_button.disabled = False

    def run_delete(button):
        global selected_df

        with output_area:
            if selected_df is None:
                print("Please preview the selected files first.")
                return

            if confirmation_widget.value.strip() != "Yes":
                print("\nOperation cancelled. No files were deleted.")
                return

            clear_output()

            print("========================================")
            print("STARTING DELETION")
            print("========================================\n")

            delete_button.disabled = True
            delete_files(selected_df)

    preview_button.on_click(preview_selection)
    delete_button.on_click(run_delete)

    display(
        HTML("<h3>Select what you want to delete</h3>"),
        folder_widget,
        preview_button,
        HTML("<hr>"),
        confirmation_widget,
        delete_button,
        output_area
    )